# 01 — Análisis Exploratorio de Datos (EDA) y Calidad de Cartera
**Credit Policy Optimizer** — Plataforma de Riesgo Crediticio y Decisión Financiera

### Objetivos del Experimento:
1. Perfilamiento estadístico y dimensional de la cartera de solicitudes crediticias sintética.
2. Estudio de las distribuciones subyacentes: Ingreso mensual, DTI, mora histórica y utilización.
3. Evaluación de correlaciones y detección de multicolinealidad.
4. Análisis bivariado con **Weight of Evidence (WoE)** e **Information Value (IV)** frente al impago (`default_flag`).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética institucional
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

from credit_policy_optimizer.data.generator import PortfolioSimulator


## 1. Generación y Carga de la Cartera
Utilizamos el simulador estocástico calibrado con covarianzas y distribuciones realistas.


In [ ]:
simulator = PortfolioSimulator(seed=42)
df = simulator.simulate(n_samples=25_000)

print(f"Dimensiones del dataset: {df.shape[0]:,} solicitudes x {df.shape[1]} columnas")
df.head(5)


## 2. Inspección Estadística y Resumen de Variables


In [ ]:
numeric_cols = [
    "monthly_income", "debt_to_income", "revolving_utilization",
    "historical_delinquencies", "loan_amount", "loan_term_months",
    "interest_rate", "cost_of_funds", "pd"
]

summary = df.select(numeric_cols).describe()
summary


### Tasa Global de Impago (Bad Rate)


In [ ]:
default_counts = df["default_flag"].value_counts()
bad_rate = df["default_flag"].mean()

print(f"Total Solicitudes: {df.height:,}")
print(f"Tasa de Impago Empírica: {bad_rate * 100:.2f}%")
default_counts


## 3. Distribución de Variables de Riesgo Clave
Analizamos el comportamiento de las variables financieras más críticas del solicitante.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Ingreso Mensual (Log-Normal)
sns.histplot(df["monthly_income"].to_numpy(), kde=True, ax=axes[0, 0], color="#2b5c8f", bins=40)
axes[0, 0].set_title("Distribución de Ingreso Mensual (USD)")
axes[0, 0].set_xlabel("Ingreso Mensual ($)")

# 2. Relación Deuda-Ingreso (DTI)
sns.histplot(df["debt_to_income"].to_numpy(), kde=True, ax=axes[0, 1], color="#d95f02", bins=40)
axes[0, 1].set_title("Distribución de Debt-to-Income (DTI)")
axes[0, 1].set_xlabel("DTI Ratio")

# 3. Utilización de Líneas de Crédito
sns.histplot(df["revolving_utilization"].to_numpy(), kde=True, ax=axes[1, 0], color="#7570b3", bins=40)
axes[1, 0].set_title("Utilización de Línea de Crédito (Revolving)")
axes[1, 0].set_xlabel("Ratio de Utilización")

# 4. Moras Históricas (Poisson)
sns.countplot(x=df["historical_delinquencies"].to_numpy(), ax=axes[1, 1], color="#e7298a")
axes[1, 1].set_title("Moras Previas en los Últimos 24 Meses (30+ DPD)")
axes[1, 1].set_xlabel("Número de Eventos de Mora")

plt.tight_layout()
plt.show()


## 4. Matriz de Correlación y Multicolinealidad
Evaluamos el grado de colinealidad entre los predictores financieros.


In [ ]:
corr_cols = [
    "monthly_income", "debt_to_income", "revolving_utilization",
    "historical_delinquencies", "loan_amount", "interest_rate", "default_flag"
]
corr_matrix = df.select(corr_cols).to_pandas().corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap="vlag", fmt=".2f", vmin=-1, vmax=1, center=0)
plt.title("Matriz de Correlación de Pearson")
plt.show()


## 5. Análisis de Poder Predictivo: Weight of Evidence (WoE) e Information Value (IV)

El **Information Value (IV)** mide el poder de separación univariado de un predictor frente a la variable binaria de incumplimiento:
$$WoE_i = \ln\left(\frac{\% \text{Goods}_i}{\% \text{Bads}_i}\right)$$
$$IV = \sum_{i=1}^k (\% \text{Goods}_i - \% \text{Bads}_i) \times WoE_i$$

*Criterio de referencia de la industria bancaria:*
- $< 0.02$: No predictivo
- $0.02 - 0.10$: Predictor débil
- $0.10 - 0.30$: Predictor medio
- $0.30 - 0.50$: Predictor fuerte
- $> 0.50$: Muy fuerte / sospechoso de fuga de información (*target leakage*)


In [ ]:
def calculate_woe_iv(df_pl: pl.DataFrame, feature: str, target: str = "default_flag", bins: int = 10):
    pdf = df_pl.select([feature, target]).to_pandas()
    
    try:
        pdf["bin"] = pd.qcut(pdf[feature], q=bins, duplicates="drop")
    except Exception:
        pdf["bin"] = pd.cut(pdf[feature], bins=bins)
        
    grouped = pdf.groupby("bin", observed=True)[target].agg(["count", "sum"])
    grouped.columns = ["total", "bads"]
    grouped["goods"] = grouped["total"] - grouped["bads"]
    
    total_goods = float(grouped["goods"].sum())
    total_bads = float(grouped["bads"].sum())
    
    grouped["dist_goods"] = grouped["goods"] / total_goods
    grouped["dist_bads"] = grouped["bads"] / total_bads
    
    # Evitar divisiones por cero
    grouped["dist_goods"] = np.where(grouped["dist_goods"] == 0, 1e-4, grouped["dist_goods"])
    grouped["dist_bads"] = np.where(grouped["dist_bads"] == 0, 1e-4, grouped["dist_bads"])
    
    grouped["woe"] = np.log(grouped["dist_goods"] / grouped["dist_bads"])
    grouped["iv_component"] = (grouped["dist_goods"] - grouped["dist_bads"]) * grouped["woe"]
    
    iv = float(grouped["iv_component"].sum())
    return iv, grouped

features_to_test = [
    "debt_to_income", "revolving_utilization", "historical_delinquencies",
    "monthly_income", "loan_amount", "interest_rate"
]

iv_summary = []
for feat in features_to_test:
    iv_val, _ = calculate_woe_iv(df, feat)
    iv_summary.append({"feature": feat, "information_value": round(iv_val, 4)})

iv_df = pl.DataFrame(iv_summary).sort("information_value", descending=True)
print("=== Ranking de Information Value (IV) ===")
print(iv_df)


In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(x="information_value", y="feature", data=iv_df.to_pandas(), palette="viridis")
plt.axvline(x=0.30, color="red", linestyle="--", label="Umbral Fuerte (IV = 0.30)")
plt.axvline(x=0.10, color="orange", linestyle="--", label="Umbral Medio (IV = 0.10)")
plt.title("Poder Predictivo por Predictor (Information Value)")
plt.xlabel("Information Value (IV)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 6. Conclusiones y Hallazgos para el Pipeline de Modelado
1. **Predictores dominantes**: `debt_to_income` y `revolving_utilization` presentan los IV más elevados, confirmando su rol esencial en la probabilidad de default.
2. **Moras históricas**: `historical_delinquencies` muestra un comportamiento escalonado con fuerte penalización ante 1 o más eventos.
3. **Puntos de mejora para `FeatureEngineer`**:
   - Términos de interacción (ej. `revolving_utilization * debt_to_income`).
   - La relación cuota-ingreso (`loan_to_income`) captura riesgo incremental más allá del monto absoluto.
